In [ ]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
class Custom2DBraTSDataset(Dataset):
    def __init__(self, data_dir, modality):
        self.data_dir = data_dir
        self.modality = modality
        self.patient_ids = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

        # Initialize lists to store slices
        self.images = []
        self.labels = []
        self.skipped_files = []
        
        # Iterate through patients and load slices
        for patient_id in self.patient_ids:
            patient_path = os.path.join(self.data_dir, patient_id)
            try:
            # Load image and label volumes
                image = nib.load(os.path.join(patient_path, f'{patient_id}_{self.modality}.nii.gz')).get_fdata()
                label = nib.load(os.path.join(patient_path, f'{patient_id}_seg.nii.gz')).get_fdata()
            except:
                print(f'Skipped file due to an error:{patient_id}_{self.modality}')
                self.skipped_files.append(os.path.join(patient_path, f'{patient_id}_{self.modality}.nii.gz'))
                continue
            # Append all slices to the list
            for slice_idx in range(image.shape[2] // 2 - 20, image.shape[2] // 2 + 20):
                image_slice = image[:, :, slice_idx]
                label_slice = label[:, :, slice_idx]

                # Convert to torch tensor and add channel dimension for image
                image_tensor = torch.tensor(image_slice, dtype=torch.float32).unsqueeze(0)  # Add channel dimension
                # rgb_image_tensor = torch.cat((image_tensor, image_tensor, image_tensor), dim=0)
                label_tensor = torch.tensor(label_slice, dtype=torch.long)

                self.images.append(image_tensor)
                self.labels.append(label_tensor)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        return image, label

    def __len__(self):
        return len(self.images)

def save_dataset(dataset, filepath):
    images = []
    labels = []
    for i in range(len(dataset)):
        img, lbl = dataset[i]
        images.append(img.numpy())  # Convert to numpy array for storage
        labels.append(lbl.numpy())
    data = {'images': images, 'labels': labels}
    torch.save(data, filepath)

class LoadDatasetFromDisk(Dataset):
    def __init__(self, filepath):
        data = torch.load(filepath)
        self.images = [torch.tensor(image) for image in data['images']]
        self.labels = [torch.tensor(label) for label in data['labels']]

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

    def __len__(self):
        return len(self.images)

# Define data directory and modalities
data_dir = '../data/BraTS_2018_Train'
modalities = ['t1']

# Load datasets for each modality
datasets = [Custom2DBraTSDataset(data_dir, mod) for mod in modalities]
print(len(datasets))
# Combine datasets using ConcatDataset

# Save the concatenated dataset
save_dataset(datasets, '../data/BraTS_2018_Training_dataset/training_t1_dataset.pth')

# Example of loading the dataset
# loaded_dataset = LoadDatasetFromDisk('../data/BraTS_2018_Training_dataset/training_dataset.pth')
# dataloader = DataLoader(loaded_dataset, batch_size=4, shuffle=True)

In [ ]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset


class LoadDatasetFromDisk():
    def __init__(self, filepath):
        data = torch.load(filepath)
        self.images = [torch.tensor(image) for image in data['images']]
        self.labels = [torch.tensor(label) for label in data['labels']]

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

    def __len__(self):
        return len(self.images)


# 각 데이터셋 로드
loaded_t1_dataset = LoadDatasetFromDisk('../data/BraTS_2018_Training_dataset/training_dataset_t1.pth')
loaded_t2_dataset = LoadDatasetFromDisk('../data/BraTS_2018_Training_dataset/training_dataset_t2.pth')
loaded_t1ce_dataset = LoadDatasetFromDisk('../data/BraTS_2018_Training_dataset/training_dataset_t1ce.pth')
loaded_flair_dataset = LoadDatasetFromDisk('../data/BraTS_2018_Training_dataset/training_dataset_flair.pth')

# 모든 데이터셋을 하나로 합치기
combined_dataset = ConcatDataset([loaded_t1_dataset, loaded_t2_dataset, loaded_t1ce_dataset, loaded_flair_dataset])

# 데이터셋 길이 확인 및 DataLoader 설정
print(f"Combined dataset length: {len(combined_dataset)}")
dataloader = DataLoader(combined_dataset, batch_size=4, shuffle=True)